In [0]:
# ML Model: Canadian Electronics Retail Sales Forecast
#XGBoost regression on gold_ml_training_dataset. Predicts next month's ca_retail_elec_sales.

In [0]:
%pip install xgboost

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb
import mlflow
import mlflow.xgboost

TARGET = "ca_retail_elec_sales"
TEST_MONTHS = 12
EXPERIMENT_NAME = "/Shared/canada_electronics_retail_forecast"

print("✅ Setup complete")

In [0]:
df = spark.table("canada_business.gold.gold_ml_training_dataset").toPandas()
df["month_date"] = pd.to_datetime(df["month_date"])
df = df.sort_values("month_date").reset_index(drop=True)

print(f"Rows: {len(df)} | Range: {df['month_date'].min().date()} → {df['month_date'].max().date()}")
print(f"Target non-null: {df[TARGET].notna().sum()}")

In [0]:
drop_cols = [
    "ingest_ts",
    "inventory_health",
    "inventory_recommendation",
    "napcs_categories_count",
    "demand_supply_gap",
    "tariff_sentiment_interaction",
    "import_to_sales_ratio",
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print(f"✅ Dropped {len(drop_cols)} columns → {len(df.columns)} remaining")

In [0]:
IMPORTS_STALE_DATE = pd.Timestamp("2023-09-01")
for col in ["ca_elec_imports", "ca_elec_imports_lag1m", "ca_elec_imports_lag3m"]:
    if col in df.columns:
        df.loc[df["month_date"] > IMPORTS_STALE_DATE, col] = np.nan

print(f"✅ Nulled stale imports after {IMPORTS_STALE_DATE.date()}")

In [0]:
df["month_sin"] = np.sin(2 * np.pi * df["month_date"].dt.month / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month_date"].dt.month / 12)
print("✅ Seasonality features added")

In [0]:
df_model = df[df[TARGET].notna()].copy()
print(f"✅ Rows with target: {len(df_model)} ({df_model['month_date'].min().date()} → {df_model['month_date'].max().date()})")

In [0]:
FINAL_FEATURES = [
    # Lag features (autoregressive — strongest predictors)
    "ca_retail_elec_sales_lag1m",
    "ca_retail_elec_sales_lag12m",
    # Macro
    "oil_wti",
    "fx_cad_usd",
    "rate_10y",
    "rate_fedfunds",
    "consumer_sentiment",
    "housing_starts",
    "real_disposable_income",
    "ppi_electronics",
    # Supply
    "inventory_ratio",
    "elec_tariff_rate",
    # Seasonality
    "month_sin",
    "month_cos",
]
print(f"✅ Features selected: {len(FINAL_FEATURES)}")

In [0]:
avail = df_model[FINAL_FEATURES].notna().mean().sort_values()
print("Feature availability (% non-null):")
for f, v in avail.items():
    flag = "⚠️" if v < 0.80 else "✅"
    print(f"  {flag} {f}: {v:.1%}")

In [0]:
corr = df_model[FINAL_FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print("Correlation with target (absolute):")
for f, v in corr.items():
    print(f"  {f}: {v:.3f}")

In [0]:
df_ml = df_model[["month_date"] + FINAL_FEATURES + [TARGET]].dropna(subset=FINAL_FEATURES + [TARGET])

X = df_ml[FINAL_FEATURES].values
y = df_ml[TARGET].values
dates = df_ml["month_date"].values

split_idx = len(df_ml) - TEST_MONTHS
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
dates_train, dates_test = dates[:split_idx], dates[split_idx:]

print(f"Train: {len(X_train)} rows ({pd.Timestamp(dates_train[0]).date()} → {pd.Timestamp(dates_train[-1]).date()})")
print(f"Test:  {len(X_test)} rows ({pd.Timestamp(dates_test[0]).date()} → {pd.Timestamp(dates_test[-1]).date()})")

In [0]:
tscv = TimeSeriesSplit(n_splits=5)
cv_results = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
    model_cv = xgb.XGBRegressor(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=1.0, reg_lambda=5.0, min_child_weight=5, random_state=42,
    )
    model_cv.fit(X_train[train_idx], y_train[train_idx],
                 eval_set=[(X_train[val_idx], y_train[val_idx])], verbose=False)
    
    y_cv_pred = model_cv.predict(X_train[val_idx])
    mae = mean_absolute_error(y_train[val_idx], y_cv_pred)
    mape = mean_absolute_percentage_error(y_train[val_idx], y_cv_pred) * 100
    cv_results.append({"fold": fold+1, "mae": mae, "mape": mape})
    print(f"Fold {fold+1}: MAE={mae:,.0f}  MAPE={mape:.2f}%")

cv_df = pd.DataFrame(cv_results)
print(f"\nAvg CV MAE: {cv_df['mae'].mean():,.0f} | Avg CV MAPE: {cv_df['mape'].mean():.2f}%")

In [0]:
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=f"xgb_{datetime.now().strftime('%Y%m%d_%H%M')}") as run:
    model = xgb.XGBRegressor(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=1.0, reg_lambda=5.0, min_child_weight=5, random_state=42,
    )
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = mean_absolute_percentage_error(y_test, y_pred) * 100
    r2 = r2_score(y_test, y_pred)

    mlflow.log_params({"model": "XGBRegressor", "features": len(FINAL_FEATURES),
                        "train_rows": len(X_train), "test_rows": len(X_test)})
    mlflow.log_metrics({"mae": mae, "rmse": rmse, "mape": mape, "r2": r2,
                         "cv_avg_mape": cv_df["mape"].mean()})
    mlflow.xgboost.log_model(model, "model")

    print(f"✅ Logged to MLflow: {run.info.run_id}")
    print(f"\nTEST RESULTS ({TEST_MONTHS} months)")
    print(f"  MAE:  {mae:,.0f} CAD")
    print(f"  RMSE: {rmse:,.0f} CAD")
    print(f"  MAPE: {mape:.2f}%")
    print(f"  R²:   {r2:.4f}")

In [0]:
importance = pd.DataFrame({"feature": FINAL_FEATURES, "importance": model.feature_importances_})
importance = importance.sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance["feature"], importance["importance"], color="#4a90d9")
ax.set_xlabel("Importance (Gain)")
ax.set_title("Feature Importance — XGBoost Retail Sales Forecast")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [3, 1]})

ax1 = axes[0]
ax1.plot(pd.to_datetime(dates_train), y_train, color="#888", linewidth=1, label="Train")
ax1.plot(pd.to_datetime(dates_test), y_test, color="#2c3e50", linewidth=2, label="Actual")
ax1.plot(pd.to_datetime(dates_test), y_pred, color="#e74c3c", linewidth=2, linestyle="--", marker="o", markersize=4, label="Predicted")
ax1.axvspan(pd.to_datetime(dates_test[0]), pd.to_datetime(dates_test[-1]), alpha=0.08, color="red")
ax1.set_ylabel("Retail Sales (CAD)")
ax1.set_title("Canadian Electronics Retail Sales — Actual vs Predicted", fontsize=14, fontweight="bold")
ax1.legend()
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1e9:.1f}B"))
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

ax2 = axes[1]
residuals = y_test - y_pred
ax2.bar(pd.to_datetime(dates_test), residuals, color=["#27ae60" if r > 0 else "#e74c3c" for r in residuals], width=25)
ax2.axhline(y=0, color="black", linewidth=0.5)
ax2.set_ylabel("Residual")
ax2.set_title("Residuals (Actual - Predicted)")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1e6:.0f}M"))
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [0]:
results = pd.DataFrame({
    "month": pd.to_datetime(dates_test),
    "actual_B": y_test / 1e9,
    "predicted_B": y_pred / 1e9,
    "error_M": (y_test - y_pred) / 1e6,
    "pct_error": ((y_pred - y_test) / y_test) * 100,
}).round(2)

display(results)

In [0]:
pred_df = pd.DataFrame({
    "month_date": pd.to_datetime(dates_test),
    "actual_sales_cad": y_test,
    "predicted_sales_cad": y_pred,
    "residual_cad": y_test - y_pred,
    "pct_error": ((y_pred - y_test) / y_test) * 100,
    "model_version": run.info.run_id,
    "prediction_ts": datetime.now(),
})
spark.createDataFrame(pred_df).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("canada_business.gold.gold_ml_predictions")

print("✅ Predictions saved to canada_business.gold.gold_ml_predictions")

In [0]:
latest = df_ml.iloc[-1:]
X_next = latest[FINAL_FEATURES].values
next_pred = model.predict(X_next)[0]
next_month = latest["month_date"].iloc[0] + pd.DateOffset(months=1)

print(f"Forecast for: {next_month.strftime('%B %Y')}")
print(f"Predicted:    ${next_pred/1e9:.3f}B CAD")
print(f"Last actual:  ${latest[TARGET].iloc[0]/1e9:.3f}B CAD")
print(f"Change:       {((next_pred - latest[TARGET].iloc[0]) / latest[TARGET].iloc[0]) * 100:.1f}%")

forecast_df = pd.DataFrame({
    "forecast_month": [next_month], "predicted_sales_cad": [next_pred],
    "based_on_month": [latest["month_date"].iloc[0]],
    "model_version": [run.info.run_id], "forecast_ts": [datetime.now()],
})
spark.createDataFrame(forecast_df).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("canada_business.gold.gold_ml_forecast")

print("✅ Forecast saved to canada_business.gold.gold_ml_forecast")